# Traveling Salesman Problem — Solution (ported from `Solution.cpp`)

Task (`task.md`):

1. Create a set of cities (points) with coordinates `x, y` on a plane with height `z`.
   Cost A→B = Euclidean distance, if a road exists. Scenarios:
   - **a.** all direct connections / ~80% of possible connections
   - **b.** symmetrical / asymmetrical (asymmetrical: uphill +10%, downhill −10%)
   - Coordinates randomly from `x, y ∈ <-100, 100>`, `z ∈ <0, 50>`.
2. Represent the map as a weighted (directed) graph (cities = nodes, roads = edges).
3. Solve TSP: start from a chosen city, visit every city exactly once, return to start, minimize cost.
   State = partial/full path from starting city. Search = state tree.
   - **a.** full search: BFS and DFS
   - **b.** greedy: NN and Dijkstra
   - **c.** A* with inadmissible / admissible heuristics
   - **d.** ACO
4. Test each algorithm in each scenario for `n = 5…20` cities: path cost, time, memory.

> All algorithms are implemented here: DFS/BFS (§3a–3b), NN + Dijkstra-greedy (§3c), admissible/inadmissible A* (§3d), ACO (§3e), with the §4 benchmark. (`Solution.cpp` still has only DFS; the rest are C++ stubs.)

## 1. Setup — data model

In [1]:
import gc
import math
import random
import time
import tracemalloc
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Tuple

# --- RNG (mirrors random_xy(-100, 100), random_z(0, 50) in Solution.cpp) ---
rng = random.Random()

@dataclass
class City:
    x: int
    y: int
    z: int

@dataclass
class AlgorithmStats:
    time_ns: int = 0
    memory_usage: int = 0  # measured peak Python bytes via tracemalloc (see measure_peak)
    total_cost: float = float("inf")
    best_path: List[int] = field(default_factory=list)

def format_bytes(n) -> str:
    """Human-readable byte count, e.g. 3288 -> '3.2 KB', 101698736 -> '97.0 MB'."""
    if n == "" or n is None:
        return ""
    n = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}" if unit != "B" else f"{int(n)} B"
        n /= 1024

def printout_stats(title: str, stats: AlgorithmStats) -> None:
    print(f"{title} -\tTime = {stats.time_ns} ns\tMemory Usage = {format_bytes(stats.memory_usage)} "
          f"({stats.memory_usage} B)\t"
          f"Total Cost = {stats.total_cost}\nPath = {stats.best_path}")

class Graph:
    def __init__(self) -> None:
        self.cities: List[City] = []
        self.paths: Dict[Tuple[int, int], float] = {}  # (src, dst) -> cost; directed

    def path_exists(self, source_id: int, dest_id: int) -> bool:
        return (source_id, dest_id) in self.paths

# SolutionAlgorithm = Callable[[Graph], AlgorithmStats]  (mirrors C++ using alias)
SolutionAlgorithm = Callable[[Graph], AlgorithmStats]

def euclidean_distance(a: City, b: City) -> float:
    dx = a.x - b.x
    dy = a.y - b.y
    dz = a.z - b.z
    return math.sqrt(dx * dx + dy * dy + dz * dz)

def measure_peak(fn: SolutionAlgorithm, graph: Graph, **kwargs) -> Tuple[AlgorithmStats, int]:
    """Run fn a second time under tracemalloc; return (stats, peak_bytes).

    Peak = max Python-object memory allocated during the run (tracemalloc peak),
    not process RSS — comparable across algorithms. Callers should take timing
    from an untraced run and overwrite stats.memory_usage with the peak here."""
    gc.collect()
    tracemalloc.stop()
    tracemalloc.start()
    try:
        stats = fn(graph, **kwargs)
        _, peak = tracemalloc.get_traced_memory()
    finally:
        tracemalloc.stop()
    return stats, int(peak)

## 2. Graph creation — scenarios

In [ ]:
def create_graph(city_count: int, is_complete: bool, is_symetrical: bool) -> Graph:
    g = Graph()
    for _ in range(city_count):
        x = rng.randint(-100, 100)
        y = rng.randint(-100, 100)
        z = 0 if is_symetrical else rng.randint(0, 50)
        g.cities.append(City(x, y, z))

    n = len(g.cities)
    for i in range(n):
        for j in range(n):
            if i == j or g.path_exists(i, j):
                continue
            city_i = g.cities[i]
            city_j = g.cities[j]
            distance = euclidean_distance(city_i, city_j)
            if not is_symetrical and city_i.z != city_j.z:
                uphill = city_i.z < city_j.z  # i -> j goes up
                if uphill:
                    g.paths[(i, j)] = distance * 1.1
                    g.paths[(j, i)] = distance * 0.9
                else:
                    g.paths[(i, j)] = distance * 0.9
                    g.paths[(j, i)] = distance * 1.1
            else:
                g.paths[(i, j)] = distance
                g.paths[(j, i)] = distance

    if not is_complete:
        removed: set[Tuple[int, int]] = set()
        target = int(0.2 * len(g.paths))
        while len(removed) < target:
            i = rng.randint(0, city_count - 1)
            j = rng.randint(0, city_count - 1)
            if i == j:
                continue
            removed.add((i, j))
            removed.add((j, i))
        for key in removed:
            g.paths.pop(key, None)
    return g

## 3a. Full search — DFS.

In [3]:
def dfs(graph: Graph) -> AlgorithmStats:
    n = len(graph.cities)
    assert n > 0
    stats = AlgorithmStats()

    best_cost = float("inf")
    max_depth = 0
    visited = [False] * n
    path: List[int] = []
    best_path: List[int] = []

    def visit(current: int, cost: float) -> None:
        nonlocal best_cost, max_depth, best_path
        max_depth = max(max_depth, len(path))
        # Prune (assumes non-negative edge costs).
        if cost >= best_cost:
            return
        if len(path) == n:
            back = graph.paths.get((current, 0))
            if back is not None:
                total = cost + back
                if total < best_cost:
                    best_cost = total
                    best_path = list(path)
            return
        for nxt in range(n):
            if visited[nxt]:
                continue
            edge = graph.paths.get((current, nxt))
            if edge is None:
                continue
            visited[nxt] = True
            path.append(nxt)
            visit(nxt, cost + edge)
            path.pop()
            visited[nxt] = False

    start = time.perf_counter_ns()
    visited[0] = True
    path.append(0)
    visit(0, 0.0)
    end = time.perf_counter_ns()
    stats.time_ns = end - start

    SIZEOF_INT = 4
    SIZEOF_DOUBLE = 8
    approx_frame = SIZEOF_INT + SIZEOF_DOUBLE + 64
    # Fallback analytic estimate; the §4 benchmark overwrites this with the tracemalloc peak.
    stats.memory_usage = n // 8 + 2 * n * SIZEOF_INT + (max_depth + 1) * approx_frame

    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats

## 3b. Full search — BFS

Exhaustive breadth-first search over the state tree (level-order expansion of partial paths from city 0).
Each state is `(current, visited_mask, path, cost)`; the queue starts with `([0], 0)` and expands all successors
one depth level at a time. Branch-and-bound pruning (`cost >= best_cost`) keeps optimality (edge costs are
non-negative). First solution found is NOT necessarily optimal (unlike DFS which dives deep immediately),
so the whole queue is drained (modulo pruning) and `best_cost` tracks the optimum — exactly like `dfs`,
but in BFS order. In-function fallback memory figure = peak queue length × estimated state size;
the §4 benchmark replaces it with the measured tracemalloc peak (see `measure_peak`).

In [4]:
from collections import deque

def bfs(graph: Graph) -> AlgorithmStats:
    """Exhaustive BFS over the TSP state tree (level-order), with branch-and-bound pruning."""
    n = len(graph.cities)
    assert n > 0
    stats = AlgorithmStats()
    best_cost = float("inf")
    best_path: List[int] = []
    max_queue = 0
    full_mask = (1 << n) - 1

    start = time.perf_counter_ns()
    queue: deque = deque()
    queue.append((0, 1 << 0, [0], 0.0))  # (current, visited_mask, path, cost)
    while queue:
        max_queue = max(max_queue, len(queue))
        current, mask, path, cost = queue.popleft()
        if cost >= best_cost:
            continue  # prune (non-negative edge costs)
        if mask == full_mask:
            back = graph.paths.get((current, 0))
            if back is not None:
                total = cost + back
                if total < best_cost:
                    best_cost = total
                    best_path = list(path)
            continue
        for nxt in range(n):
            if mask & (1 << nxt):
                continue
            edge = graph.paths.get((current, nxt))
            if edge is None:
                continue
            new_cost = cost + edge
            if new_cost >= best_cost:
                continue
            queue.append((nxt, mask | (1 << nxt), path + [nxt], new_cost))
    end = time.perf_counter_ns()
    stats.time_ns = end - start

    SIZEOF_INT = 4
    state_bytes = 8 + 8 + n * SIZEOF_INT + 28  # mask + cost + path copy + overhead
    stats.memory_usage = max_queue * state_bytes + n * SIZEOF_INT

    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats

## 3c. Greedy search — Nearest Neighbour and Dijkstra-assisted greedy

- **Nearest Neighbour (NN):** from the current city always go to the cheapest directly-connected unvisited city.
  Fails (no tour) on incomplete graphs when the current city has no direct road to any unvisited city.
- **Dijkstra-greedy (`dijkstra`, correctly-spelled alias `dijkstra`):** same greedy idea, but the next city is the
  *nearest unvisited city by shortest-path distance* — each step runs Dijkstra from the current city and follows
  the shortest path to that target. Transit cities on the path count as visited, so this works on incomplete
  graphs by routing through intermediate cities. Note: `best_path` here is the full walk *including* transit
  revisits and the final return to 0 (length ≥ n+1), unlike the other methods which list n cities in visit order.

In [5]:
import heapq

def nearest_neighbour(graph: Graph) -> AlgorithmStats:
    """Greedy NN: repeatedly visit the cheapest directly-reachable unvisited city."""
    n = len(graph.cities)
    assert n > 0
    stats = AlgorithmStats()
    start = time.perf_counter_ns()
    visited = [False] * n
    path: List[int] = [0]
    visited[0] = True
    current = 0
    cost = 0.0
    feasible = True
    for _ in range(n - 1):
        best = None
        best_edge = float("inf")
        for nxt in range(n):
            if visited[nxt]:
                continue
            edge = graph.paths.get((current, nxt))
            if edge is not None and edge < best_edge:
                best_edge = edge
                best = nxt
        if best is None:  # dead end (incomplete graph)
            feasible = False
            break
        visited[best] = True
        path.append(best)
        cost += best_edge
        current = best
    if feasible:
        back = graph.paths.get((current, 0))
        if back is None:
            feasible = False
        else:
            cost += back
    end = time.perf_counter_ns()
    stats.time_ns = end - start
    stats.memory_usage = n * 4 + n * 1 + 64
    if feasible:
        stats.total_cost = round(cost)
        stats.best_path = path
    return stats

def _dijkstra_sssp(graph: Graph, src: int):
    """Standard Dijkstra from src over the road graph. Returns (dist, prev)."""
    n = len(graph.cities)
    dist = [float("inf")] * n
    prev = [-1] * n
    dist[src] = 0.0
    pq = [(0.0, src)]
    done = [False] * n
    while pq:
        d, u = heapq.heappop(pq)
        if done[u]:
            continue
        done[u] = True
        for v in range(n):
            e = graph.paths.get((u, v))
            if e is None:
                continue
            nd = d + e
            if nd < dist[v] - 1e-12:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    return dist, prev

def _reconstruct_path(prev: List[int], src: int, dst: int) -> List[int]:
    seg: List[int] = []
    c = dst
    while c != -1:
        seg.append(c)
        if c == src:
            break
        c = prev[c]
    seg.reverse()
    return seg if seg and seg[0] == src else []

def dijikstra(graph: Graph) -> AlgorithmStats:  # keep C++ typo in name
    """Greedy TSP via repeated Dijkstra (nearest unvisited city by shortest-path distance)."""
    n = len(graph.cities)
    assert n > 0
    stats = AlgorithmStats()
    start = time.perf_counter_ns()
    visited = [False] * n
    visited[0] = True
    remaining = n - 1
    walk: List[int] = [0]
    current = 0
    cost = 0.0
    feasible = True
    while remaining > 0:
        dist, prev = _dijkstra_sssp(graph, current)
        target = -1
        best_d = float("inf")
        for v in range(n):
            if not visited[v] and dist[v] < best_d:
                best_d = dist[v]
                target = v
        if target == -1 or not math.isfinite(best_d):
            feasible = False
            break
        seg = _reconstruct_path(prev, current, target)
        if not seg:
            feasible = False
            break
        for node in seg[1:]:
            edge = graph.paths.get((walk[-1], node))
            if edge is None:
                feasible = False
                break
            cost += edge
            walk.append(node)
            if not visited[node]:
                visited[node] = True
                remaining -= 1
        current = target
        if not feasible:
            break
    if feasible:
        dist, prev = _dijkstra_sssp(graph, current)
        if not math.isfinite(dist[0]):
            feasible = False
        else:
            seg = _reconstruct_path(prev, current, 0)
            for node in seg[1:]:
                cost += graph.paths[(walk[-1], node)]
                walk.append(node)
    end = time.perf_counter_ns()
    stats.time_ns = end - start
    stats.memory_usage = n * n * 8 + n * 4 + len(walk) * 4 + 256
    if feasible:
        stats.total_cost = round(cost)
        stats.best_path = walk
    return stats

def dijkstra(graph: Graph) -> AlgorithmStats:
    return dijikstra(graph)  # correctly-spelled alias

## 3d. A* — admissible vs inadmissible heuristics

A* over the same state tree: priority `f = g + h`, `g` = cost so far, `h` = estimated cost to finish the tour.
Closed set `best_g[(current, visited_mask)]` prunes dominated states. The first full tour popped from the
heap is returned — optimal if `h` is admissible (never overestimates), suboptimal otherwise.

- **Admissible `h`:** `MST(unvisited ∪ {current}) + min(return to 0)`.
  Any feasible completion must connect those nodes (≥ MST, using `min(i→j, j→i)` so asymmetry can't inflate it)
  and must return to 0 (≥ cheapest return edge). `inf` when disconnected. Never overestimates ⇒ optimal A*.
- **Inadmissible `h`:** greedy NN completion cost from the state, inflated ×1.5 (a feasible completion is an
  *upper* bound on the optimum, so it overestimates ⇒ A* becomes a fast approximation). Dead ends fall back
  to `(remaining+1) × max_edge × 1.5`.

In [6]:
def _mst_cost(nodes, graph: Graph) -> float:
    """Prim MST over `nodes` with w(i,j) = min(cost(i->j), cost(j->i)); inf if disconnected."""
    nodes = list(nodes)
    if len(nodes) <= 1:
        return 0.0
    in_mst = {nodes[0]}
    total = 0.0
    best: Dict[int, float] = {}
    for v in nodes[1:]:
        a = graph.paths.get((nodes[0], v))
        b = graph.paths.get((v, nodes[0]))
        cands = [c for c in (a, b) if c is not None]
        best[v] = min(cands) if cands else float("inf")
    while len(in_mst) < len(nodes):
        u = min((v for v in nodes if v not in in_mst), key=lambda v: best[v], default=None)
        if u is None or not math.isfinite(best[u]):
            return float("inf")
        total += best[u]
        in_mst.add(u)
        for v in nodes:
            if v in in_mst:
                continue
            a = graph.paths.get((u, v))
            b = graph.paths.get((v, u))
            cands = [c for c in (a, b) if c is not None]
            if cands:
                best[v] = min(best[v], min(cands))
    return total

def _h_admissible(graph: Graph, current: int, unvisited) -> float:
    if not unvisited:
        return graph.paths.get((current, 0), float("inf"))
    mst = _mst_cost(set(unvisited) | {current}, graph)
    if not math.isfinite(mst):
        return float("inf")
    ret = min((graph.paths.get((u, 0), float("inf")) for u in unvisited), default=float("inf"))
    return mst + ret

def _h_inadmissible(graph: Graph, current: int, unvisited, max_edge: float) -> float:
    if not unvisited:
        return graph.paths.get((current, 0), float("inf"))
    rem = set(unvisited)
    cur = current
    total = 0.0
    while rem:
        bu = None
        bc = float("inf")
        for u in rem:
            e = graph.paths.get((cur, u))
            if e is not None and e < bc:
                bc = e
                bu = u
        if bu is None:  # dead end -> inflated fallback
            return (len(rem) + 1) * max_edge * 1.5
        total += bc
        cur = bu
        rem.remove(bu)
    back = graph.paths.get((cur, 0))
    if back is None:
        return (len(unvisited) + 1) * max_edge * 1.5
    return (total + back) * 1.5

def _astar(graph: Graph, heuristic) -> AlgorithmStats:
    n = len(graph.cities)
    assert n > 0
    stats = AlgorithmStats()
    full_mask = (1 << n) - 1
    start = time.perf_counter_ns()
    h0 = heuristic(graph, 0, {i for i in range(1, n)})
    pq = [(h0, 0.0, 0, 1 << 0, (0,))]  # (f, g, current, mask, path)
    best_g: Dict[tuple, float] = {}
    max_pq = 1
    best_cost = float("inf")
    best_path: List[int] = []
    while pq:
        max_pq = max(max_pq, len(pq))
        f, g, cur, mask, path = heapq.heappop(pq)
        if g >= best_g.get((cur, mask), float("inf")):
            continue
        best_g[(cur, mask)] = g
        if mask == full_mask:
            back = graph.paths.get((cur, 0))
            if back is not None:
                best_cost = g + back
                best_path = list(path)
                break  # first goal popped = A* answer (optimal only for admissible h)
            continue
        if f >= best_cost:
            continue
        unvis = {i for i in range(n) if not (mask >> i & 1)}
        for nxt in unvis:
            edge = graph.paths.get((cur, nxt))
            if edge is None:
                continue
            ng = g + edge
            if ng >= best_g.get((nxt, mask | (1 << nxt)), float("inf")):
                continue
            h = heuristic(graph, nxt, unvis - {nxt})
            if not math.isfinite(h):
                continue
            heapq.heappush(pq, (ng + h, ng, nxt, mask | (1 << nxt), path + (nxt,)))
    end = time.perf_counter_ns()
    stats.time_ns = end - start
    stats.memory_usage = len(best_g) * 24 + max_pq * 80 + n * 4
    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats

def admissible_astar(graph: Graph) -> AlgorithmStats:
    """A* with admissible MST heuristic (optimal, still exponential worst-case)."""
    return _astar(graph, _h_admissible)

def inadmissible_astar(graph: Graph) -> AlgorithmStats:
    """A* with inadmissible inflated-NN heuristic (fast approximation)."""
    max_edge = max(graph.paths.values()) if graph.paths else float("inf")
    return _astar(graph, lambda g, c, u: _h_inadmissible(g, c, u, max_edge))

## 3e. Ant Colony Optimization (ACO)

Classic Dorigo ACO for TSP, adapted to directed/incomplete graphs: `n_ants` ants per iteration (default
`max(10, n)`, 100 iterations) start at city 0 and repeatedly pick the next unvisited city with probability
proportional to `tau^alpha * eta^beta` (`tau` = pheromone, `eta = 1/cost`). Only direct roads are candidates,
so ants can get stuck on incomplete graphs (that tour is discarded). After each iteration pheromones evaporate
(`× (1-rho)`) and successful ants deposit `Q / tour_cost`. The best tour over all iterations is returned.
Defaults: `alpha=1.0, beta=3.0, rho=0.3, Q=100.0`. Pass `seed=` for reproducibility.

In [7]:
def ant_colony(graph: Graph, n_ants=None, n_iter: int = 100, alpha: float = 1.0,
               beta: float = 3.0, rho: float = 0.3, Q: float = 100.0, seed=None) -> AlgorithmStats:
    n = len(graph.cities)
    assert n > 0
    stats = AlgorithmStats()
    local_rng = random.Random(seed)
    start = time.perf_counter_ns()
    if n == 1:
        stats.time_ns = 0
        stats.total_cost = 0
        stats.best_path = [0]
        stats.memory_usage = 64
        return stats
    tau = [[1.0] * n for _ in range(n)]
    best_cost = float("inf")
    best_path: List[int] = []
    n_ants = n_ants or max(10, n)
    for _ in range(n_iter):
        tours = []
        for _ in range(n_ants):
            cur = 0
            vis = [False] * n
            vis[0] = True
            path = [0]
            cost = 0.0
            ok = True
            for _ in range(n - 1):
                feas = [v for v in range(n) if not vis[v] and (cur, v) in graph.paths]
                if not feas:
                    ok = False
                    break
                weights = []
                total = 0.0
                for v in feas:
                    e = graph.paths[(cur, v)]
                    eta = 1.0 / e if e > 0 else 1e9
                    w = (tau[cur][v] ** alpha) * (eta ** beta)
                    weights.append(w)
                    total += w
                r = local_rng.random() * total
                acc = 0.0
                chosen = feas[-1]
                for v, w in zip(feas, weights):
                    acc += w
                    if r <= acc:
                        chosen = v
                        break
                cost += graph.paths[(cur, chosen)]
                cur = chosen
                vis[cur] = True
                path.append(cur)
            if ok:
                back = graph.paths.get((cur, 0))
                if back is None:
                    ok = False
                else:
                    cost += back
            if ok:
                tours.append((cost, path))
                if cost < best_cost:
                    best_cost = cost
                    best_path = list(path)
        for i in range(n):
            row = tau[i]
            for j in range(n):
                row[j] *= (1.0 - rho)
        for cost, path in tours:
            deposit = Q / cost if cost > 0 else Q
            prev = path[0]
            for node in path[1:]:
                tau[prev][node] += deposit
                prev = node
            tau[prev][0] += deposit
    end = time.perf_counter_ns()
    stats.time_ns = end - start
    stats.memory_usage = n * n * 8 + n_ants * n * 4 + n * 4 + 256
    if math.isfinite(best_cost):
        stats.total_cost = round(best_cost)
        stats.best_path = best_path
    return stats


## 4. Experiments — `n = 5…20`, 4 scenarios

Task §4: test each algorithm in each scenario (complete/incomplete × symmetrical/asymmetrical) for
`n = 5…20` cities: path cost, time, memory. Exact state-tree searches (`dfs`, `bfs`, `admissible_astar`)
are exponential, so they run only up to `EXACT_MAX_N = 10` (larger `n` recorded as `SKIPPED`);
greedy / inadmissible-A* / ACO run the full range. `rng.seed(...)` keeps graphs reproducible.
Note `dijkstra`-greedy `best_path` is the full walk incl. transit revisits + final return, all others list
the n cities in visit order (cost always includes the return to 0). Every row is also saved to
`results.csv` (columns: `n, scenario, algorithm, status, total_cost, time_ns, memory_usage, memory_human, best_path`).
`memory_usage` holds exact peak bytes (numeric, for analysis); `memory_human` is the same value formatted
as KB/MB via `format_bytes` (same text shown in the printed `Memory Usage` field).

In [ ]:
EXACT_MAX_N = 10  # dfs / bfs / admissible A* only run up to this n (exponential)
EXACT_IDS = {"dfs", "bfs", "admissible_astar"}
NAMED_SOLS: List[Tuple[str, SolutionAlgorithm]] = [
    ("dfs", dfs),
    ("bfs", bfs),
    ("nearest_neighbour", nearest_neighbour),
    ("dijkstra", dijkstra),
    ("inadmissible_astar", inadmissible_astar),
    ("admissible_astar", admissible_astar),
    ("ant_colony", ant_colony),
]
SCENARIOS = [
    ("complete symmetrical   ", True, True),
    ("complete asymmetrical ", True, False),
    ("incomplete symmetrical", False, True),
    ("incomplete asymmetrical", False, False),
]
RESULTS_CSV = "results.csv"

import csv

rng.seed(42)
rows = []
for i in range(5, 21):
    print(f"=== n={i} ===")
    for title, is_complete, is_sym in SCENARIOS:
        g = create_graph(i, is_complete, is_sym)
        scenario = title.strip()
        for name, fn in NAMED_SOLS:
            if name in EXACT_IDS and i > EXACT_MAX_N:
                print(f"{name:20s} {title}: SKIPPED (n > {EXACT_MAX_N})")
                rows.append({"n": i, "scenario": scenario, "algorithm": name,
                             "status": "skipped", "total_cost": "", "time_ns": "",
                             "memory_usage": "", "memory_human": "", "best_path": ""})
                continue
            kwargs = {"seed": 1} if name == "ant_colony" else {}
            # Pass 1 (untraced): authoritative time + tour. Pass 2 (traced): peak memory.
            # Tracing slows allocation, so timing is never taken under tracemalloc.
            st = fn(g, **kwargs)  # type: ignore[call-arg]
            _, peak = measure_peak(fn, g, **kwargs)
            st.memory_usage = peak
            printout_stats(f"{name:20s} {title}", st)
            ok = math.isfinite(st.total_cost)
            rows.append({"n": i, "scenario": scenario, "algorithm": name,
                         "status": "ok" if ok else "no_tour",
                         "total_cost": st.total_cost if ok else "",
                         "time_ns": st.time_ns, "memory_usage": st.memory_usage,
                         "memory_human": format_bytes(st.memory_usage),
                         "best_path": "-".join(map(str, st.best_path)) if ok else ""})
    print()

with open(RESULTS_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["n", "scenario", "algorithm", "status",
                                          "total_cost", "time_ns", "memory_usage", "memory_human",
                                          "best_path"])
    writer.writeheader()
    writer.writerows(rows)
print(f"Saved {len(rows)} rows to {RESULTS_CSV}")


=== n=5 ===
dfs                  complete symmetrical    -	Time = 31489 ns	Memory Usage = 496 B	Total Cost = 548
Path = [0, 3, 2, 1, 4]
bfs                  complete symmetrical    -	Time = 44523 ns	Memory Usage = 1556 B	Total Cost = 548
Path = [0, 3, 2, 1, 4]
nearest_neighbour    complete symmetrical    -	Time = 6512 ns	Memory Usage = 89 B	Total Cost = 566
Path = [0, 4, 2, 3, 1]
dijkstra             complete symmetrical    -	Time = 53070 ns	Memory Usage = 500 B	Total Cost = 566
Path = [0, 4, 2, 3, 1, 0]
inadmissible_astar   complete symmetrical    -	Time = 57679 ns	Memory Usage = 700 B	Total Cost = 553
Path = [0, 1, 2, 3, 4]
admissible_astar     complete symmetrical    -	Time = 218550 ns	Memory Usage = 1772 B	Total Cost = 548
Path = [0, 3, 2, 1, 4]
ant_colony           complete symmetrical    -	Time = 6014892 ns	Memory Usage = 676 B	Total Cost = 548
Path = [0, 4, 1, 2, 3]
dfs                  complete asymmetrical  -	Time = 43913 ns	Memory Usage = 496 B	Total Cost = 644
Path = [0, 1, 